# 🌿 Garden Guardian — Phase 1 Training

Train YOLOv8n to detect **your cultivated crops**. Everything else in the garden is a removal candidate.

**Before running:**
1. Label your photos in [Roboflow](https://roboflow.com) (format: *YOLOv8*)
2. Download & unzip the export into `data/` so you have `data/train`, `data/valid`, `data/test`
3. Check that `data/data.yaml` class names match your Roboflow classes

In [ ]:
%pip install -q ultralytics
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Sanity-check the dataset
Catches image/label mismatches and class imbalance **before** wasting a training run.

In [ ]:
import sys; sys.path.append('../src')
from utils import dataset_summary
for split in ('../data/train', '../data/valid', '../data/test'):
    dataset_summary(split)

## 2. Train
`yolov8n` (nano) is intentional — trains in <1h on a gaming laptop and will later fit on edge hardware (Phase 3).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # pretrained on COCO, we fine-tune
results = model.train(
    data='../data/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,      # lower to 8 if you hit CUDA out-of-memory
    patience=20,   # early stop if no improvement
)

## 3. Evaluate

In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

## 4. Test on a real photo
This produces your **README demo image** — pick the best-looking result and copy it to `docs/demo.jpg`.

In [ ]:
res = model.predict(source='../data/test/images', conf=0.5, save=True)
print('Annotated images saved under runs/detect/predict*/')

## Notes
- **conf=0.5 is a safety choice**: a crop the model is unsure about must never be treated as a weed downstream.
- Best weights: `runs/detect/train/weights/best.pt` → upload to **GitHub Releases**, not the repo.
- If accuracy is weak: add more photos in different lighting / angles before tuning hyperparameters.